In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt


In [ ]:
#Load and Prepare Data
df = pd.read_csv('helmet_sensor_dataset_realistic.csv')

feature_cols = [
    'Location_Latitude', 'Location_Longitude', 'Temperature', 'Humidity',
    'Wind_Speed', 'Precipitation', 'Weather_Condition', 'Visibility',
    'Traffic_Speed', 'Congestion_Level', 'Road_Type', 'Travel_Time_Estimate'
]

df_routes = df.dropna(subset=feature_cols).copy()

df_routes['Is_Risky'] = (
    (df_routes['Impact_Detected'] == 1) |
    (df_routes['Fusion_Risk_Alert'] == 'Drowsy') |
    (df_routes['Fusion_Risk_Alert'] == 'Impact') |
    (df_routes['Incident_Type'].notna())
).astype(int)

print(f"Routes for training: {len(df_routes)}")
print(f"Risky routes: {df_routes['Is_Risky'].sum()} ({df_routes['Is_Risky'].mean()*100:.1f}%)")

categorical_cols = ['Weather_Condition', 'Congestion_Level', 'Road_Type']
numerical_cols = [c for c in feature_cols if c not in categorical_cols]

df_cat = pd.get_dummies(df_routes[categorical_cols])

scaler = StandardScaler()
df_num_scaled = pd.DataFrame(
    scaler.fit_transform(df_routes[numerical_cols]),
    columns=numerical_cols,
    index=df_routes.index
)

X_pd = pd.concat([df_num_scaled, df_cat], axis=1)
X = X_pd.values.astype(np.float32)
y = df_routes['Is_Risky'].values.astype(np.float32)

# Save preprocessing info for inference
cat_columns = df_cat.columns.tolist()
num_means = scaler.mean_
num_scales = scaler.scale_


In [ ]:
#Train Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
#Define and Train Keras Model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# Plot training history
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {test_acc*100:.2f}%')


In [ ]:
# ── Confusion Matrix & Classification Metrics ──────────────────────────────

# Get predicted probabilities and binary predictions
y_prob = model.predict(X_test).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Safe', 'Risky'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized (recall per class)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=['Safe', 'Risky'])
disp_norm.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

# --- Classification Report ---
print("\n── Classification Report ──────────────────────────────────────────────")
print(classification_report(y_test, y_pred, target_names=['Safe', 'Risky']))

# --- Key Derived Metrics ---
precision  = tp / (tp + fp) if (tp + fp) > 0 else 0
recall     = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1         = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
roc_auc    = roc_auc_score(y_test, y_prob)

print("── Derived Metrics ────────────────────────────────────────────────────")
print(f"  True Positives  (TP): {tp}")
print(f"  True Negatives  (TN): {tn}")
print(f"  False Positives (FP): {fp}  ← Safe misclassified as Risky")
print(f"  False Negatives (FN): {fn}  ← Risky misclassified as Safe")
print(f"  Precision  : {precision:.4f}")
print(f"  Recall     : {recall:.4f}")
print(f"  Specificity: {specificity:.4f}")
print(f"  F1-Score   : {f1:.4f}")
print(f"  ROC-AUC    : {roc_auc:.4f}")

# --- ROC Curve ---
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
#Direct Conversion to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable default optimizations (dynamic-range quantization: int8 weights, float32 inputs/outputs)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save the TFLite model
with open('route_risk_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model saved! Size: {len(tflite_model) / 1024:.1f} KB")


In [ ]:
#Predict for New Route + Safety Suggestions
interpreter = tf.lite.Interpreter(model_path='route_risk_model.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def predict_route_risk_tflite(route_dict):
    input_df = pd.DataFrame([route_dict])

    input_cat = pd.get_dummies(input_df[categorical_cols])
    input_cat = input_cat.reindex(columns=cat_columns, fill_value=0)

    input_num = (input_df[numerical_cols].values - num_means) / num_scales
    input_num = pd.DataFrame(input_num, columns=numerical_cols)

    input_processed = np.hstack([input_num.values, input_cat.values]).astype(np.float32)

    # Run inference
    interpreter.set_tensor(input_details[0]['index'], input_processed)
    interpreter.invoke()
    prob = interpreter.get_tensor(output_details[0]['index'])[0][0]

    if prob < 0: prob = 0
    if prob > 1: prob = 1

    is_risky = prob > 0.5
    status = "Risky" if is_risky else "Safe"
    print(f"Predicted Risk Probability: {prob*100:.1f}% → {status}")

    # Safety suggestions
    if is_risky:
        suggestions = ["***General Advice:*** Wear your helmet properly, avoid distractions, and stay hydrated."]

        weather = route_dict['Weather_Condition']
        if weather in ['Rainy', 'Snowy', 'Foggy']:
            suggestions.append("• Adverse weather: Reduce speed, increase following distance, and use appropriate lights.")

        if route_dict.get('Precipitation', 0) > 0.5:
            suggestions.append("• Precipitation detected: Roads may be slippery — brake gently.")

        if route_dict.get('Visibility', 10000) < 5000:
            suggestions.append("• Low visibility: Drive slowly and use headlights.")

        cong = route_dict['Congestion_Level']
        if cong == 'High':
            suggestions.append("• High congestion: Anticipate sudden stops and maintain safe distance.")

        road = route_dict['Road_Type']
        if road in ['Narrow paths', 'Heavy traffic', 'Busy urban', 'Poor lighting road']:
            suggestions.append(f"• Challenging road ({road}): Stay extra vigilant; consider a safer alternative route.")

        if route_dict.get('Travel_Time_Estimate', 0) > 90:
            suggestions.append("• Long travel time: Plan regular breaks to avoid fatigue.")

        print("\n***Safety Recommendations for This Route:***")
        for s in suggestions:
            print(s)
    else:
        print("\nRoute appears Safe. Still follow basic safety: Wear helmet, obey traffic rules, and ride defensively.")


In [ ]:
example_route = {
    'Location_Latitude': 40.71,
    'Location_Longitude': -74.01,
    'Temperature': 15.0,
    'Humidity': 80.0,
    'Wind_Speed': 20.0,
    'Precipitation': 1.2,
    'Weather_Condition': 'Rainy',
    'Visibility': 5000.0,
    'Traffic_Speed': 30.0,
    'Congestion_Level': 'High',
    'Road_Type': 'Busy urban',
    'Travel_Time_Estimate': 120.0
}

predict_route_risk_tflite(example_route)
